In [41]:
import pandas as pd
import requests
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import glob
import os

# force to show all rows and columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns

In [42]:
# variables to find race and driver, also cut off time to take out yellow flags and pit stop laps
race_id = 5552
driver_id = 0
lap_time_cutoff = 74

race_name = 'Mexico' + '_250.csv'
path = './data/16_Mexico/'
driver = 'Shane Van Gisbergen'
filtered_lap_value = 125
filtered_speed_value = 60

In [43]:
driver_csv = path + driver + '.csv'
df = pd.read_csv(driver_csv)
df = df.drop([11,21])
print(df.head(5))

# check lap_value to use
annotation = 'filtered: ' + str(filtered_speed_value) + 'mph'
fig = px.line(df, x='Lap', y='LapSpeed', title='SPEED')
fig.add_hline(y=filtered_speed_value, line_width=0.5, line_color='red', annotation_text=annotation)
fig.show()

# check speed_value to use
annotation = 'filtered: ' + str(filtered_lap_value) + 'sec'
fig = px.line(df, x='Lap', y='LapTime', title='LAP TIMES')
fig.add_hline(y=filtered_lap_value, line_width=0.5, line_color='red', annotation_text=annotation)
fig.show()

del df

   Unnamed: 0  Lap  LapTime  LapSpeed  RunningPos  Number Manufacturer
0           0    0      NaN       NaN           1      88          Chv
1           1    1  110.321    78.970           1      88          Chv
2           2    2  199.709    43.623           1      88          Chv
3           3    3  206.629    42.163           4      88          Chv
4           4    4  204.452    42.611           5      88          Chv


In [44]:
# find race name and location
url = 'https://cf.nascar.com/cacher/2025/race_list_basic.json'
json = requests.get(url).json()
race = pd.json_normalize(json['series_1'])
race = race[['race_id', 'track_name', 'race_name']].set_index('race_id')
race = race.loc[race_id]
location = race.values[0]
name = race.values[1]
race = name + ' @ ' + location

# find list of race ids
race_list = pd.json_normalize(json['series_1'])
race_list = race_list[['race_id', 'track_name', 'race_name']].set_index('race_id')

print(race_list)

                                   track_name  \
race_id                                         
5543                      Bowman Gray Stadium   
5544           Daytona International Speedway   
5545           Daytona International Speedway   
5546           Daytona International Speedway   
5547                   Atlanta Motor Speedway   
5551                  Circuit of The Americas   
5549                          Phoenix Raceway   
5548                 Las Vegas Motor Speedway   
5583                 Homestead-Miami Speedway   
5553                    Martinsville Speedway   
5558                       Darlington Raceway   
5550                   Bristol Motor Speedway   
5555                  Talladega Superspeedway   
5554                     Texas Motor Speedway   
5557                          Kansas Speedway   
5561                North Wilkesboro Speedway   
5562                North Wilkesboro Speedway   
5563                 Charlotte Motor Speedway   
5568                

In [45]:
# pull driver list from api
url = f'https://cf.nascar.com/cacher/2025/1/{race_id}/lap-times.json'
json = requests.get(url).json()
driver_list = pd.json_normalize(json['laps'])
driver_list = driver_list[['FullName','Number','Manufacturer','RunningPos']]

In [47]:
for i in np.arange(len(driver_list)):
    
    # Get data into dataframe
    url = f'https://cf.nascar.com/cacher/2025/1/{race_id}/lap-times.json'
    json = requests.get(url).json()
    lap_times = pd.json_normalize(json['laps'][i]['Laps'])
    
    # Find driver name to see who we're dealing with
    driver_info = pd.DataFrame(json['laps'][i])
    driver_name = driver_info['FullName'][0]
    driver_name = driver_name.lstrip('* ').rstrip(' #(iFTC)')
    lap_times['Number'] = driver_info['Number'][0]
    lap_times['Manufacturer'] = driver_info['Manufacturer'][0]
    
    # Save to .csv
    driver_name_csv = path + driver_name + '.csv'
    lap_times.to_csv(driver_name_csv)

    print(driver_name)

print(lap_times.head())

Shane Van Gisbergen
Christopher Bell
Chase Elliott
Alex Bowman
Michael McDowell
John Hunter Nemechek
Chase Briscoe
Cole Custer
William Byron
Chris Buescher
Ty Gibbs
Bubba Wallace
AJ Allmendinger
Ryan Blaney
Ryan Preece
Ross Chastain
Erik Jones
Austin Cindric
Daniel Suarez
Tyler Reddick
Joey Logano
Todd Gilliland
Ryan Truex
Justin Haley
Brad Keselowsk
Josh Berry
Ricky Stenhouse Jr.
Austin Dillon
Riley Herbst
Noah Gragson
Cody Ware
Katherine Legge
Ty Dillon
Carson Hocevar
Zane Smith
Kyle Larson
Kyle Busch
   Lap  LapTime LapSpeed  RunningPos Number Manufacturer
0    0      NaN     None          11      8          Chv
1    1  120.381   72.370          10      8          Chv
2    2  212.373   41.022          16      8          Chv
3    3  191.086   45.592          11      8          Chv
4    4  198.135   43.970          12      8          Chv


In [48]:
print(driver_list)

                 FullName Number Manufacturer  RunningPos
0   Shane Van Gisbergen #     88          Chv           1
1        Christopher Bell     20          Tyt           2
2           Chase Elliott      9          Chv           3
3             Alex Bowman     48          Chv           4
4        Michael McDowell     71          Chv           5
5    John Hunter Nemechek     42          Tyt           6
6           Chase Briscoe     19          Tyt           7
7             Cole Custer     41          Frd           8
8           William Byron     24          Chv           9
9          Chris Buescher     17          Frd          10
10               Ty Gibbs     54          Tyt          11
11          Bubba Wallace     23          Tyt          12
12        AJ Allmendinger     16          Chv          13
13            Ryan Blaney     12          Frd          14
14            Ryan Preece     60          Frd          15
15          Ross Chastain      1          Chv          16
16            